# 🧠 Local LLM Quiz Generator — Development Notebook

This notebook walks through building the quiz generator using **Ollama (gemma4)** + **LangChain**.

It demonstrates:
- Strict guardrails via system prompt
- Positive testing (fact retrieval from quiz bank)
- Negative testing (refusal on out-of-domain topics like "Rome")

All code here is kept in sync with `app.py` and `test_assistant.py`.

> **Requirements**: Make sure you have run `pip install -r requirements.txt` in your activated `.venv`.


## 0. Setup & Environment


In [1]:
# Ensure all dependencies are installed in the current kernel
import subprocess, sys
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "langchain>=0.3.0", "langchain-core>=0.3.0",
    "langchain-ollama>=0.2.0", "ollama>=0.4.0",
    "python-dotenv>=1.0.0"
])


import warnings
warnings.filterwarnings('ignore')

import os
from dotenv import load_dotenv, find_dotenv

_ = load_dotenv(find_dotenv())
print("dotenv loaded (safe if no .env file)")


dotenv loaded (safe if no .env file)


## 1. Imports (Modern LangChain + Ollama)


In [2]:

from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import OllamaLLM
from langchain_core.output_parsers import StrOutputParser


## 2. Quiz Bank + System Prompt (with Guardrails)

This is the **exact same** `quiz_bank` and `system_message` used in `app.py`.
The prompt contains explicit refusal rules for categories not in the bank.


In [3]:

quiz_bank = """1. Subject: Leonardo DaVinci
   Categories: Art, Science
   Facts:
    - Painted the Mona Lisa
    - Studied zoology, anatomy, geology, optics
    - Designed a flying machine

2. Subject: Paris
   Categories: Art, Geography
   Facts:
    - Location of the Louvre, the museum where the Mona Lisa is displayed
    - Capital of France
    - Most populous city in France
    - Where Radium and Polonium were discovered by scientists Marie and Pierre Curie

3. Subject: Telescopes
   Category: Science
   Facts:
    - Device to observe different objects
    - The first refracting telescopes were invented in the Netherlands in the 17th Century
    - The James Webb space telescope is the largest telescope in space. It uses a gold-berillyum mirror

4. Subject: Starry Night
   Category: Art
   Facts:
    - Painted by Vincent van Gogh in 1889
    - Captures the east-facing view of van Gogh's room in Saint-Rémy-de-Provence

5. Subject: Physics
   Category: Science
   Facts:
    - The sun doesn't change color during sunset.
    - Water slows the speed of light
    - The Eiffel Tower in Paris is taller in the summer than the winter due to expansion of the metal.
"""


In [4]:

delimiter = "####"

system_message = f"""
Follow these steps to generate a customized quiz for the user.
The question will be delimited with four hashtags i.e {delimiter}

The user will provide a category that they want to create a quiz for. Any questions included in the quiz
should only refer to the category.

Step 1:{delimiter} First identify the category user is asking about from the following list:
* Geography
* Science
* Art

Step 2:{delimiter} Determine the subjects to generate questions about. The list of topics are below:

{quiz_bank}

Pick up to two subjects that fit the user's category. 

Step 3:{delimiter} Generate a quiz for the user. Based on the selected subjects generate 3 questions for the user using the facts about the subject.

Use the following format for the quiz:
Question 1:{delimiter} <question 1>

Question 2:{delimiter} <question 2>

Question 3:{delimiter} <question 3>

Additional rules:

- Only use explicit matches for the category, if the category is not an exact match to categories in the quiz bank, answer that you do not have information.
- If the user asks a question about a subject you do not have information about in the quiz bank, answer "I'm sorry I do not have information about that".
"""


## 3. The Reusable `assistant_chain`

This helper builds the LangChain pipeline:
`ChatPromptTemplate` → `OllamaLLM` → `StrOutputParser`

It accepts an optional `system_message` so we can reuse it for tests.


In [5]:

def assistant_chain(
    system_message=system_message,
    human_template="{question}",
    llm=None,
    output_parser=None):

  if llm is None:
    llm = OllamaLLM(model="gemma4:latest", temperature=0)
  if output_parser is None:
    output_parser = StrOutputParser()

  chat_prompt = ChatPromptTemplate.from_messages([
      ("system", system_message),
      ("human", human_template),
  ])
  return chat_prompt | llm | output_parser


## 4. Quick Manual Test (Positive)


In [6]:

assistant = assistant_chain()
result = assistant.invoke({"question": "Generate a quiz about art."})
print(result)


Question 1:#### What famous painting did Leonardo DaVinci paint?

Question 2:#### Who painted the artwork "Starry Night"?

Question 3:#### Besides painting, what subjects did Leonardo DaVinci study?


## 5. Automated Evaluation Functions

These match exactly the helpers in `test_assistant.py`.

- `eval_expected_words`: positive test — the output must mention certain keywords from the quiz bank.
- `evaluate_refusal`: negative test — the model must politely refuse topics outside the bank.


In [7]:

def eval_expected_words(
    system_message,
    question,
    expected_words,
    human_template="{question}",
    llm=None,
    output_parser=None):

  if llm is None:
    llm = OllamaLLM(model="gemma4:latest", temperature=0)
  if output_parser is None:
    output_parser = StrOutputParser()

  assistant = assistant_chain(system_message, human_template, llm, output_parser)
  answer = assistant.invoke({"question": question})
  print(answer)

  assert any(word in answer.lower() for word in expected_words),     f"Expected the assistant questions to include '{expected_words}', but it did not"


In [8]:

def evaluate_refusal(
    system_message,
    question,
    decline_response,
    human_template="{question}", 
    llm=None,
    output_parser=None):

  if llm is None:
    llm = OllamaLLM(model="gemma4:latest", temperature=0)
  if output_parser is None:
    output_parser = StrOutputParser()

  assistant = assistant_chain(system_message, human_template, llm, output_parser)
  answer = assistant.invoke({"question": question})
  print(answer)

  assert decline_response.lower() in answer.lower(),     f"Expected the bot to decline with '{decline_response}' got {answer}"


## 6. Run the Automated Tests (inline)


In [9]:

# Positive test - Science
question  = "Generate a quiz about science."
expected_subjects = ["davinci", "telescope", "physics", "curie"]
eval_expected_words(system_message, question, expected_subjects)


Question 1:#### What subjects did Leonardo DaVinci study, besides painting the Mona Lisa?

Question 2:#### According to the facts, what causes the Eiffel Tower to be taller in the summer than in the winter?

Question 3:#### What scientific fact is mentioned regarding the speed of light and water, or the color of the sun at sunset?


In [10]:

# Positive test - Geography
question  = "Generate a quiz about geography."
expected_subjects = ["paris", "france", "louvre"]
eval_expected_words(system_message, question, expected_subjects)


Question 1:#### What major scientific elements, Radium and Polonium, were discovered by Marie and Pierre Curie in the city of Paris?

Question 2:#### Paris is known as the capital of which European country?

Question 3:#### Besides being the most populous city in France, Paris is also the location of the Louvre, which houses the Mona Lisa.


In [11]:

# Negative / Refusal test
question  = "Help me create a quiz about Rome"
decline_response = "I'm sorry"
evaluate_refusal(system_message, question, decline_response)


I'm sorry I do not have information about that


## 7. Write the Final `app.py` and `test_assistant.py`

Running these cells will overwrite the module files with the exact code that was validated in this notebook.
This keeps everything in sync.


In [12]:
%%writefile app.py
from langchain_core.prompts                import ChatPromptTemplate
from langchain_ollama                    import OllamaLLM
from langchain_core.output_parsers   import StrOutputParser

delimiter = "####"

quiz_bank = """1. Subject: Leonardo DaVinci
   Categories: Art, Science
   Facts:
    - Painted the Mona Lisa
    - Studied zoology, anatomy, geology, optics
    - Designed a flying machine

2. Subject: Paris
   Categories: Art, Geography
   Facts:
    - Location of the Louvre, the museum where the Mona Lisa is displayed
    - Capital of France
    - Most populous city in France
    - Where Radium and Polonium were discovered by scientists Marie and Pierre Curie

3. Subject: Telescopes
   Category: Science
   Facts:
    - Device to observe different objects
    - The first refracting telescopes were invented in the Netherlands in the 17th Century
    - The James Webb space telescope is the largest telescope in space. It uses a gold-berillyum mirror

4. Subject: Starry Night
   Category: Art
   Facts:
    - Painted by Vincent van Gogh in 1889
    - Captures the east-facing view of van Gogh's room in Saint-Rémy-de-Provence

5. Subject: Physics
   Category: Science
   Facts:
    - The sun doesn't change color during sunset.
    - Water slows the speed of light
    - The Eiffel Tower in Paris is taller in the summer than the winter due to expansion of the metal.
"""

system_message = f"""
Follow these steps to generate a customized quiz for the user.
The question will be delimited with four hashtags i.e {delimiter}

The user will provide a category that they want to create a quiz for. Any questions included in the quiz
should only refer to the category.

Step 1:{delimiter} First identify the category user is asking about from the following list:
* Geography
* Science
* Art

Step 2:{delimiter} Determine the subjects to generate questions about. The list of topics are below:

{quiz_bank}

Pick up to two subjects that fit the user's category. 

Step 3:{delimiter} Generate a quiz for the user. Based on the selected subjects generate 3 questions for the user using the facts about the subject.

Use the following format for the quiz:
Question 1:{delimiter} <question 1>

Question 2:{delimiter} <question 2>

Question 3:{delimiter} <question 3>

Additional rules:

- Only use explicit matches for the category, if the category is not an exact match to categories in the quiz bank, answer that you do not have information.
- If the user asks a question about a subject you do not have information about in the quiz bank, answer "I'm sorry I do not have information about that".
"""

"""
  Helper functions for writing the test cases
"""

def assistant_chain(
    system_message=system_message,
    human_template="{question}",
    llm=None,
    output_parser=None):

  if llm is None:
    llm = OllamaLLM(model="gemma4:latest", temperature=0)
  if output_parser is None:
    output_parser = StrOutputParser()

  chat_prompt = ChatPromptTemplate.from_messages([
      ("system", system_message),
      ("human", human_template),
  ])
  return chat_prompt | llm | output_parser


Overwriting app.py


In [13]:

# Verify what was written
with open("app.py", "r", encoding="utf-8") as f:
    print(f.read())


from langchain_core.prompts                import ChatPromptTemplate
from langchain_ollama                    import OllamaLLM
from langchain_core.output_parsers   import StrOutputParser

delimiter = "####"

quiz_bank = """1. Subject: Leonardo DaVinci
   Categories: Art, Science
   Facts:
    - Painted the Mona Lisa
    - Studied zoology, anatomy, geology, optics
    - Designed a flying machine

2. Subject: Paris
   Categories: Art, Geography
   Facts:
    - Location of the Louvre, the museum where the Mona Lisa is displayed
    - Capital of France
    - Most populous city in France
    - Where Radium and Polonium were discovered by scientists Marie and Pierre Curie

3. Subject: Telescopes
   Category: Science
   Facts:
    - Device to observe different objects
    - The first refracting telescopes were invented in the Netherlands in the 17th Century
    - The James Webb space telescope is the largest telescope in space. It uses a gold-berillyum mirror

4. Subject: Starry Night
   Cate

In [14]:
%%writefile test_assistant.py
from app import assistant_chain
from app import system_message
from langchain_core.prompts                import ChatPromptTemplate
from langchain_ollama                    import OllamaLLM
from langchain_core.output_parsers   import StrOutputParser

import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

def eval_expected_words(
    system_message,
    question,
    expected_words,
    human_template="{question}",
    llm=None,
    output_parser=None):

  if llm is None:
    llm = OllamaLLM(model="gemma4:latest", temperature=0)
  if output_parser is None:
    output_parser = StrOutputParser()

  assistant = assistant_chain(system_message, human_template, llm, output_parser)
  answer = assistant.invoke({"question": question})
  print(answer)

  assert any(word in answer.lower() \
             for word in expected_words), \
    f"Expected the assistant questions to include \
    '{expected_words}', but it did not"

def evaluate_refusal(
    system_message,
    question,
    decline_response,
    human_template="{question}", 
    llm=None,
    output_parser=None):

  if llm is None:
    llm = OllamaLLM(model="gemma4:latest", temperature=0)
  if output_parser is None:
    output_parser = StrOutputParser()

  assistant = assistant_chain(system_message, 
                              human_template,
                              llm,
                              output_parser)

  answer = assistant.invoke({"question": question})
  print(answer)

  assert decline_response.lower() in answer.lower(), \
    f"Expected the bot to decline with \
    '{decline_response}' got {answer}"

"""
  Test cases
"""

def test_science_quiz():

  question  = "Generate a quiz about science."
  expected_subjects = ["davinci", "telescope", "physics", "curie"]
  eval_expected_words(
      system_message,
      question,
      expected_subjects)

def test_geography_quiz():
  question  = "Generate a quiz about geography."
  expected_subjects = ["paris", "france", "louvre"]
  eval_expected_words(
      system_message,
      question,
      expected_subjects)

def test_refusal_rome():
  question  = "Help me create a quiz about Rome"
  decline_response = "I'm sorry"
  evaluate_refusal(
      system_message,
      question,
      decline_response)


Overwriting test_assistant.py


In [15]:

# Verify
with open("test_assistant.py", "r", encoding="utf-8") as f:
    print(f.read())


from app import assistant_chain
from app import system_message
from langchain_core.prompts                import ChatPromptTemplate
from langchain_ollama                    import OllamaLLM
from langchain_core.output_parsers   import StrOutputParser

import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

def eval_expected_words(
    system_message,
    question,
    expected_words,
    human_template="{question}",
    llm=None,
    output_parser=None):

  if llm is None:
    llm = OllamaLLM(model="gemma4:latest", temperature=0)
  if output_parser is None:
    output_parser = StrOutputParser()

  assistant = assistant_chain(system_message, human_template, llm, output_parser)
  answer = assistant.invoke({"question": question})
  print(answer)

  assert any(word in answer.lower() \
             for word in expected_words), \
    f"Expected the assistant questions to include \
    '{expected_words}', but it did not"

def evaluate_refusal(
    system_message,

## 8. Run the Full Test Suite

This is the same command you would run from the terminal inside the `.venv`.


In [16]:

# Run pytest from inside the notebook (captures output nicely)
import subprocess
import sys
result = subprocess.run(
    [sys.executable, "-m", "pytest", "test_assistant.py", "-v", "--tb=short"],
    capture_output=True, text=True, cwd="."
)
print(result.stdout)
if result.returncode != 0:
    print("=== STDERR / FAILURES ===")
    print(result.stderr)


============================= test session starts =============================
platform win32 -- Python 3.14.5, pytest-9.0.3, pluggy-1.6.0 -- c:\Users\jains\OneDrive\Desktop\Quiz-generator-locally-\.venv\Scripts\python.exe
cachedir: .pytest_cache
rootdir: c:\Users\jains\OneDrive\Desktop\Quiz-generator-locally-
plugins: anyio-4.13.0, langsmith-0.8.8
collecting ... collected 3 items

test_assistant.py::test_science_quiz PASSED                              [ 33%]
test_assistant.py::test_geography_quiz PASSED                            [ 66%]
test_assistant.py::test_refusal_rome PASSED                              [100%]

======================== 3 passed in 73.58s (0:01:13) =========================



## 9. Next Steps & Notes

- The real production code lives in `app.py` + `test_assistant.py`.
- To run tests from terminal (recommended for CI):
  ```bash
  python -m pytest test_assistant.py -v
  ```
- All secrets / CI tokens have been removed from this notebook (they live in environment variables or your shell).
- Customize `quiz_bank` and `system_message` here, then re-run the writefile cells to update the modules.
- See [README.md](./README.md) for the full story behind local LLMOps + automated evals.

**All issues fixed:**
- Modern `langchain-ollama` imports everywhere
- Consistent `system_message` naming
- No more hardcoded GitHub/CircleCI tokens
- Proper markdown structure and explanations
- Empty cells removed
- Stale error outputs cleared
- Notebook now produces exactly the same files as the clean modules
